In [1]:
import sys
sys.path.append('../Enola')
from Enola.enola.enola import Enola
from gbstim.gb import GBCode
from gbstim.device import Device
from gbstim.bposd import BPOSD

import stim
from qiskit import QuantumCircuit

/Users/jviszlai/Documents/Work/Research/ProjectRepos/gbstim/tests/../Enola/enola/router/codegen.py:1329: SyntaxWarning: invalid escape sequence '\ '
  """change of convention. In solve() and the SMT model, a/c/r_s govern


In [2]:
def convert_stim_to_CZ_list(circ: stim.Circuit) -> tuple[QuantumCircuit, list[list[int]]]:
    cz_circuit = QuantumCircuit.from_qasm_str(circ.without_noise().to_qasm(open_qasm_version=2, skip_dets_and_obs=True))
    instruction = cz_circuit.data
    list_gate_two_qubit = []
    for ins in instruction:
        if ins.operation.num_qubits == 2:
            list_gate_two_qubit.append((ins.qubits[0]._index, ins.qubits[1]._index))
    return cz_circuit, list_gate_two_qubit


In [3]:
l = 6
m = 6
device = Device((2*m, 2*l))
code = GBCode(device, [(0, 2), (0, 1), (3, 0)], [(1, 0), (2, 0), (0, 3)], l, m)
print(f'[{code.n}, {code.k}, {code.d}]')
circ = code.stim_circ(num_rounds=code.d)
cz_circ, cz_list = convert_stim_to_CZ_list(circ)

[72, 12, 6]
Problem Name: a914d2ad726548baad52879cb6932138
Problem Type: TSP
Number of Nodes: 13
Rounded Euclidean Norm (CC_EUCLIDEAN)
CCtsp_solve_dat ...
Finding a good tour for compression ...
linkern ...
Starting Cycle: 259
   0 Steps   Best: 258   0.00 seconds
   6 Total Steps.
Best cycle length: 258
Lin-Kernighan Running Time: 0.00
LK Initial Run: 258.0
LK Run 0: 258.0
LK Run from best tour: 258.0
Time to find compression tour: 0.00 (seconds)
Set initial upperbound to 258 (from tour)
  LP Value  1: 223.000000  (0.00 seconds)
  LP Value  2: 258.000000  (0.00 seconds)
New lower bound: 258.000000
Exact lower bound: 258.000000
DIFF: 0.000000
Established Bound: 258
Optimal tour: 258
Total Time to solve TSP: 0.00


In [4]:
scalable_enola_compiler = Enola('72-12-6_gb_code', 
                       trivial_layout = True,
                       dependency=True,
                       routing_strategy="maximalis",
                       reverse_to_initial=True,
                       use_window=True)
scalable_enola_compiler.setProgram(cz_list, cz_circ.num_qubits)
scalable_enola_compiler.setArchitecture([12, 12, 12, 12])

In [5]:
program_list = scalable_enola_compiler.solve(save_file=True)

[INFO] Enola: Start Solving
[INFO] Enola: Run scheduling
[INFO] Enola: Time for scheduling: 0.0006577968597412109s
12
[INFO] Enola: Time for placement: 1.5020370483398438e-05s
[INFO] Enola: Solve for Rydberg stage 2/140.
[INFO] Enola: Solve for Rydberg stage 3/140.
[INFO] Enola: Solve for Rydberg stage 4/140.
[INFO] Enola: Solve for Rydberg stage 5/140.
[INFO] Enola: Solve for Rydberg stage 6/140.
[INFO] Enola: Solve for Rydberg stage 7/140.
[INFO] Enola: Solve for Rydberg stage 8/140.
[INFO] Enola: Solve for Rydberg stage 9/140.
[INFO] Enola: Solve for Rydberg stage 10/140.
[INFO] Enola: Solve for Rydberg stage 11/140.
[INFO] Enola: Solve for Rydberg stage 12/140.
[INFO] Enola: Solve for Rydberg stage 13/140.
[INFO] Enola: Solve for Rydberg stage 14/140.
[INFO] Enola: Solve for Rydberg stage 15/140.
[INFO] Enola: Solve for Rydberg stage 16/140.
[INFO] Enola: Solve for Rydberg stage 17/140.
[INFO] Enola: Solve for Rydberg stage 18/140.
[INFO] Enola: Solve for Rydberg stage 19/140.
[INF

In [6]:
total_duration = 0
for instr in program_list:
    if instr['type'] == 'Move':
        total_duration += instr['duration']

In [7]:
print(f'Total duration: {total_duration / 1e3} ms for d rounds')

Total duration: 832.8948013247164 ms for d rounds
